# Thermal-averaged reaction rates from S(E) or σ(E)

This notebook converts a nuclear cross-section — supplied either as an
astrophysical S-factor `S(E)` or directly as `σ(E)` — into a **primat-format
thermonuclear rate table**, with a Monte-Carlo-propagated 1σ uncertainty
column and the correct detailed-balance header.

It is the Python replacement for the Mathematica notebook
`Thermal-Average.nb`, which lived outside this repository and is superseded by
this file.

## Physics

For charged particles the cross-section is factorised into the slowly-varying
astrophysical S-factor and the Coulomb (Gamow) penetration factor:

$$\sigma(E) = \frac{S(E)}{E}\,e^{-2\pi\eta},
\qquad \eta = \alpha_{\rm FS} Z_1 Z_2 \sqrt{\frac{\mu c^2}{2E}} $$

The thermal average over a Maxwell–Boltzmann distribution at temperature $T$ is

$$N_A\langle\sigma v\rangle
  = N_A \left(\frac{8}{\pi\mu}\right)^{1/2} (k_BT)^{-3/2}
    \int_0^\infty \sigma(E)\, E\, e^{-E/k_BT}\, {\rm d}E $$

This is the energy-space form of the velocity integral used in the Mathematica
notebook; the two are identical under $E = \tfrac12\mu v^2$, but the energy
form is better conditioned near the Gamow peak.

Reference: Pitrou, Coc, Uzan & Vangioni, *Physics Reports* **04** (2018) 005
(`biblio/Pitrou_etal_PhysReptArxivVersion.pdf`), nuclear-rates section.

## Units

| Quantity | Unit |
|---|---|
| Energy `E` | MeV |
| S-factor `S(E)` | MeV·barn |
| Cross-section `σ(E)` | barn |
| Temperature `T9` | 10⁹ K |
| Output `N_A⟨σv⟩` | cm³ mol⁻¹ s⁻¹ |

## How to use this notebook

**Edit §3 only.** Everything else is machinery. In §3 you declare the
reaction, a reference label for the output filename, whether you are giving
`S(E)` or `σ(E)`, the function itself, its parameter vector and covariance,
and where to write the result. Then run all cells.

All nuclide data — masses, charges, spins, Q-values, detailed-balance
coefficients — is read from primat itself, so this notebook cannot drift from
the solver's own nuclear data.

## §2 — Nuclide data and constants (from primat)

Nothing here is hard-coded. `PRIMATConfig` reads
`primat/data/csv/nuclides.csv` (generated offline by
`generate_rates/nuclide_table.py` from the NUBASE2020 evaluation
`nubase_4.mas20.txt`), giving `(N, Z)`, mass excess in keV, and spin for
every nuclide in primat's reaction catalog.

In [ ]:
import sys
from pathlib import Path

import numpy as np

# The notebook lives in generate_rates/; primat is importable from the repo root.
REPO = Path.cwd().parent if Path.cwd().name == "generate_rates" else Path.cwd()
sys.path.insert(0, str(REPO))

from primat.config import PRIMATConfig
from primat.constants import CONST

cfg = PRIMATConfig()

# --- Unit conversions and constants, all taken from primat ------------------
ALPHA_FS = CONST.alphaem              # fine-structure constant (dimensionless)
M_U_MEV  = cfg.ma                     # atomic mass unit          [MeV]
M_E_MEV  = cfg.me                     # electron mass             [MeV]
BARN_CM2 = 1.0e-24                    # 1 barn in cm^2 (definition of the barn)

# k_B * 1e9 K expressed in MeV, i.e. kT[MeV] = MEV_PER_GK * T9.  cfg.kB is in
# erg/K and cfg.MeV is 1 MeV in erg, so the ratio converts erg -> MeV.
MEV_PER_GK = cfg.kB * 1.0e9 / cfg.MeV

# Avogadro's number is *derived*, not typed in: it is the reciprocal of the
# atomic mass unit expressed in grams (m_u[MeV] * erg/MeV / c^2).
N_A = 1.0 / (M_U_MEV * cfg.MeV / cfg.clight**2)   # [mol^-1]


def nuclear_mass_MeV(name):
    """Rest-mass energy of a nuclide, in MeV.

    This is the *nuclear* mass (bare nucleus), not the atomic mass: the Z bound
    electrons are subtracted.  It is built exactly the way
    ``primat.network_data.compute_detailed_balance_coefficients`` builds it, so
    the reduced mass used here is consistent with the reverse-rate coefficients
    written into the output file's header:

        M = A * m_u + Delta - Z * m_e

    with ``Delta`` the mass excess from ``nuclides.csv`` (stored in keV, hence
    the 1e-3 conversion to MeV).

    Args:
        name: primat nuclide key, e.g. ``"n"``, ``"p"``, ``"H2"``, ``"He4"``.

    Returns:
        Rest-mass energy in MeV.

    Example:
        >>> round(nuclear_mass_MeV("H2"), 4)     # deuteron
        1875.6128
    """
    N, Z = cfg.Nuclides[name]
    A = N + Z
    return A * M_U_MEV + cfg.NuclExcessMass[name] * 1.0e-3 - Z * M_E_MEV


def charge(name):
    """Atomic number Z of a nuclide, from primat's ``Nuclides`` table.

    Z enters the Gamow penetration factor as the product Z1*Z2; it is zero for
    the neutron, which is why neutron-induced reactions have no Coulomb barrier.

    Args:
        name: primat nuclide key, e.g. ``"He4"``.

    Returns:
        Atomic number (int).

    Example:
        >>> charge("He4"), charge("n")
        (2, 0)
    """
    return cfg.Nuclides[name][1]


def reduced_mass_MeV(n1, n2):
    """Reduced rest-mass energy mu*c^2 of the entrance channel, in MeV.

    The thermal average is an integral over the *relative* kinetic energy of
    the two reactants, whose inertia is the reduced mass
    ``mu = m1 m2 / (m1 + m2)``.  Working with mu*c^2 in MeV keeps every energy
    in the notebook in the same unit.

    Args:
        n1, n2: primat nuclide keys of the two reactants.

    Returns:
        mu*c^2 in MeV.

    Example:
        >>> round(reduced_mass_MeV("H2", "H2"), 4)   # half the deuteron mass
        937.8064
    """
    m1, m2 = nuclear_mass_MeV(n1), nuclear_mass_MeV(n2)
    return m1 * m2 / (m1 + m2)


print(f"N_A        = {N_A:.6e} mol^-1        (expect 6.022141e+23)")
print(f"MEV_PER_GK = {MEV_PER_GK:.7f} MeV/T9  (expect 0.0861733)")
print(f"M(d)       = {nuclear_mass_MeV('H2'):.4f} MeV   (expect 1875.6128)")
print(f"mu(d,d)    = {reduced_mass_MeV('H2', 'H2'):.4f} MeV   (expect 937.8064)")